# FINAL 3 TINGKAT: self-training dengan rerank (pseudo-data Krama Alus)

Pertanyaan: apakah model demo (M2M100, resep v2 `m2m_base`) bisa dinaikkan dengan **melatih ulang memakai terjemahan
Krama Alus buatannya sendiri** yang sudah disaring reranker?

Dasar: *reranking-based transductive learning* (Wu dkk./HW-TSC, IWSLT 2023): kandidat terbaik hasil rerank dipakai
sebagai data latih tambahan; C-F naik 97,49 → 99,83%. Diagnosis `final_3lv_postedit`: sisa kesalahan Krama Alus bukan
lagi soal kata (leksikon tidak pernah kena), jadi perlu metode yang mengubah model, bukan mengganti kata.

Alur:
1. **Juri dan reranker** dilatih ulang di train split (resep v2). Weights juri **disimpan** ke `judge/` supaya run
   berikutnya bisa memakai juri yang sama.
2. **Teacher** = resep v2 `m2m_base`, dilatih ulang di kernel ini (dinilai juri yang sama dengan student).
3. **Pseudo-data**: teacher menerjemahkan kalimat Indonesia ke Krama Alus, 16 kandidat beam, rerank dengan λ=2
   (λ demo). Disimpan hanya jika P(Krama Alus) dari **reranker** ≥ 0,9. Juri tidak dipakai untuk menyaring.
   - Sumber A (`train`): kalimat train yang group-nya tidak punya referensi Krama Alus.
   - Sumber B (`nusax`): kalimat Indonesia NusaX-MT (CC BY-SA), ≤ 25 kata, tidak ada di valid/test/train.
4. **Student** = resep v2 + pseudo-data (1×, tidak di-oversample): `st_train` (A) dan `st_train_nusax` (A + B).
5. Checkpoint, decoding, dan pemenang dipilih di **valid** (skor = chrF++ + 30 × ketepatan tingkat).
   Prediksi valid ikut disimpan.

Hasil 1 seed. Kalimat pseudo belum diperiksa penutur Jawa.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacrebleu", "sentencepiece"], check=False)

import glob, json, time, random, re, shutil, os, gc, urllib.request
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import numpy as np, pandas as pd, torch, sacrebleu
import torch.nn.functional as F
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

REPO, SL, TL = "facebook/m2m100_418M", "id", "jv"
EPOCHS, OS, BS, LR, WARMUP, SEED, MAXLEN, CTRL_W = 15, 3, 16, 5e-5, 0.1, 42, 64, 30.0
NBEST, LAMS = 8, [0.5, 1.0, 2.0, 4.0]
# pseudo-data
PS_NBEST, PS_LAM, PS_THR, NUSAX_MAX_WORDS = 16, 2.0, 0.9, 25
NUSAX_URL = "https://raw.githubusercontent.com/IndoNLP/nusax/main/datasets/mt/train.csv"   # CC BY-SA
VARIANTS = {"teacher": [], "st_train": ["train"], "st_train_nusax": ["train", "nusax"]}
OUT = "/kaggle/working"
dev = "cuda"
LEVELS = ["ngoko", "krama", "krama_alus"]
LID = {l: i for i, l in enumerate(LEVELS)}
KA = LID["krama_alus"]
TAG = {l: f"<{l}>" for l in LEVELS}
INIT_WORDS = {"ngoko": "kowe aku mangan lunga turu", "krama": "sampeyan kula nedha kesah tilem",
              "krama_alus": "panjenenganipun dalem dhahar tindak sare"}
T0 = time.time()
log = lambda *a: print(f"[{time.time()-T0:6.0f}s]", *a, flush=True)


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


find = lambda n: glob.glob(f"/kaggle/input/**/{n}.csv", recursive=True)[0]
tr_raw = pd.read_csv(find("train"))
def _load3(n):
    d = pd.read_csv(find(n))
    d = d[d["level"].isin(LEVELS)].copy()
    d["label"] = d["level"].map(LID)
    return d.reset_index(drop=True)
tr, va, te = (_load3(n) for n in ["train", "valid", "test"])
log("data", len(tr), len(va), len(te), torch.cuda.get_device_name(0))
tag_src = lambda ind, level: f"{TAG[level]} {ind}"
def bleu(h, r): return sacrebleu.corpus_bleu(h, [r]).score
def chrf(h, r): return sacrebleu.corpus_chrf(h, [r], word_order=2).score


def boot_ci(h, r, n=1000, seed=0):
    rng = random.Random(seed); idx = range(len(h)); vals = []
    for _ in range(n):
        s = [rng.choice(idx) for _ in idx]
        vals.append(chrf([h[i] for i in s], [r[i] for i in s]))
    vals.sort()
    return [round(vals[int(.025 * n)], 2), round(vals[int(.975 * n)], 2)]


In [ ]:
# ------------------------------------------------------------------ classifiers (resep v2)
def train_classifier(base, seed):
    ctok = AutoTokenizer.from_pretrained(base)
    cmod = AutoModelForSequenceClassification.from_pretrained(base, num_labels=len(LEVELS)).to(dev)
    seed_all(seed)
    opt = torch.optim.AdamW(cmod.parameters(), lr=5e-5)
    X, y = tr["javanese"].tolist(), tr["label"].tolist()
    for _ in range(4):
        cmod.train(); order = list(range(len(X))); random.shuffle(order)
        for i in range(0, len(order), 32):
            b = order[i:i + 32]
            enc = ctok([X[j] for j in b], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
            loss = cmod(**enc, labels=torch.tensor([y[j] for j in b], device=dev)).loss
            opt.zero_grad(); loss.backward(); opt.step()
    cmod.eval()

    def logprobs(sents):
        out = []
        for i in range(0, len(sents), 64):
            enc = ctok(list(sents[i:i + 64]), return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
            with torch.no_grad():
                out.append(F.log_softmax(cmod(**enc).logits.float(), -1).cpu())
        return torch.cat(out).numpy()

    info = {"base": base}
    for n, d in [("valid", va), ("test", te)]:
        p = logprobs(d["javanese"].tolist()).argmax(-1); y_ = d["label"].values
        info[f"acc_{n}"] = round(float((p == y_).mean()), 4)
        info[f"per_class_acc_{n}"] = {l: round(float((p[y_ == i] == i).mean()), 4) for i, l in enumerate(LEVELS)}
    log("classifier", info)
    return logprobs, ctok, cmod, info


judge_lp, judge_tok, judge_mod, judge_info = train_classifier("w11wo/javanese-distilbert-small", SEED)
judge_mod.save_pretrained(f"{OUT}/judge"); judge_tok.save_pretrained(f"{OUT}/judge")   # juri tetap untuk run berikutnya
json.dump({"levels": LEVELS, **judge_info}, open(f"{OUT}/judge/judge_info.json", "w"), indent=1)
classify = lambda s: judge_lp(s).argmax(-1).tolist()
rr_lp, rr_tok, rr_mod, rr_info = train_classifier("w11wo/javanese-roberta-small", 7)
MARK = {"ngoko": {"kowe", "aku", "awakmu"}, "krama": {"sampeyan", "kula", "menika", "punika", "boten"},
        "alus": {"panjenengan", "panjenenganipun", "dalem", "adalem", "dhahar", "tindak", "kagungan", "sare"}}
toks = lambda s: set(re.findall(r"[\w'\-]+", str(s).lower()))


In [ ]:
# ------------------------------------------------------------------ MT (disalin dari v2)
def load():
    tok = AutoTokenizer.from_pretrained(REPO, src_lang=SL, tgt_lang=TL)
    model = AutoModelForSeq2SeqLM.from_pretrained(REPO)
    tok.add_tokens(list(TAG.values()), special_tokens=True)
    model.resize_token_embeddings(len(tok))
    emb = model.get_input_embeddings().weight
    with torch.no_grad():
        for l, words in INIT_WORDS.items():
            ids = tok(words, add_special_tokens=False)["input_ids"]
            emb[tok.convert_tokens_to_ids(TAG[l])] = emb[ids].mean(0)
    bos = tok.get_lang_id(TL)
    return tok, model.to(dev), bos


def translate(tok, model, bos, srcs, beams=4, bs=48):
    model.eval(); out = [None] * len(srcs)
    order = sorted(range(len(srcs)), key=lambda i: len(srcs[i]))
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        enc = tok([srcs[j] for j in idx], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            g = model.generate(**enc, forced_bos_token_id=bos, num_beams=beams, max_new_tokens=MAXLEN)
        for j, t in zip(idx, tok.batch_decode(g, skip_special_tokens=True)):
            out[j] = t
    return out


def seq_scores(tok, model, srcs, cands, chunk=16):
    out = []
    for i in range(0, len(srcs), chunk):
        enc = tok(srcs[i:i + chunk], text_target=cands[i:i + chunk], return_tensors="pt",
                  padding=True, truncation=True, max_length=MAXLEN).to(dev)
        labels = enc.pop("labels"); labels[labels == tok.pad_token_id] = -100
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            logits = model(**enc, labels=labels).logits
        nll = F.cross_entropy(logits.float().transpose(1, 2), labels, ignore_index=-100, reduction="none")
        m = (labels != -100).float()
        out += (-(nll * m).sum(1) / m.sum(1).clamp(min=1)).cpu().tolist()
        del logits, nll
    return np.array(out)


def nbest(tok, model, bos, srcs, k=NBEST, bs=8):
    model.eval(); out = [None] * len(srcs)
    order = sorted(range(len(srcs)), key=lambda i: len(srcs[i]))
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        enc = tok([srcs[j] for j in idx], return_tensors="pt", padding=True, truncation=True, max_length=MAXLEN).to(dev)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            g = model.generate(**enc, forced_bos_token_id=bos, num_beams=k, num_return_sequences=k, max_new_tokens=MAXLEN)
        txt = tok.batch_decode(g, skip_special_tokens=True)
        sc = seq_scores(tok, model, [srcs[j] for j in idx for _ in range(k)], txt)
        for n, j in enumerate(idx):
            out[j] = (txt[n * k:(n + 1) * k], sc[n * k:(n + 1) * k])
    torch.cuda.empty_cache()
    return out


def rerank(nb, levels, lam):
    flat = [t for c, _ in nb for t in c]
    lp = rr_lp(flat); res = []; p = 0
    for (c, s), l in zip(nb, levels):
        k = len(c); tot = s + lam * lp[p:p + k, LID[l]]; p += k
        res.append(c[int(np.argmax(tot))])
    return res


def ctrl_acc(hyps, levels):
    p = np.array(classify(hyps)); y = np.array([LID[l] for l in levels])
    return float((p == y).mean()), {l: round(float((p[y == i] == i).mean()), 4) for i, l in enumerate(LEVELS)}


def valid_score(h):
    c = chrf(h, va["javanese"].tolist()); a, alv = ctrl_acc(h, va["level"].tolist())
    return c + CTRL_W * a, round(c, 2), round(a, 4), alv


def honor_pairs():
    pairs = []
    for _, g in tr_raw.groupby("group"):
        jv = dict(zip(g["level"], g["javanese"]))
        if "krama" in jv and "krama_alus" in jv and jv["krama"] != jv["krama_alus"]:
            pairs.append((f"{TAG['krama_alus']} {jv['krama']}", jv["krama_alus"]))
    return pairs


In [ ]:
# ------------------------------------------------------------------ evaluasi & training
def evaluate(name, hyp_test, ctrl_req, ctrl_out):
    res = {"test": {}}
    for lv in LEVELS + ["all"]:
        m = te["level"] == lv if lv != "all" else te["level"].notna()
        h, r = [x for x, k in zip(hyp_test, m) if k], te.loc[m, "javanese"].tolist()
        res["test"][lv] = {"n": len(r), "BLEU": round(bleu(h, r), 2), "chrF++": round(chrf(h, r), 2), "chrF++_CI95": boot_ci(h, r)}
    p = classify(ctrl_out)
    conf = np.zeros((len(LEVELS), len(LEVELS)), int)
    for (_, l), q in zip(ctrl_req, p):
        conf[LID[l], q] += 1
    u = sorted({s for s, _ in ctrl_req})
    O = {(s, l): o for (s, l), o in zip(ctrl_req, ctrl_out)}
    same = lambda a, b: round(float(np.mean([O[(s, a)] == O[(s, b)] for s in u])), 4)
    res["control"] = {"level_accuracy": round(float(np.trace(conf) / conf.sum()), 4),
                      "per_level_accuracy": {l: round(float(conf[i, i] / conf[i].sum()), 4) for i, l in enumerate(LEVELS)},
                      "confusion_rows_requested_cols_predicted": conf.tolist(),
                      "identical_rate": {"krama=krama_alus": same("krama", "krama_alus"), "ngoko=krama": same("ngoko", "krama")},
                      "markers_by_requested_level": {l: {k: sum(bool(toks(O[(s, l)]) & v) for s in u) for k, v in MARK.items()} for l in LEVELS}}
    pred = te.copy(); pred["hyp"] = hyp_test
    pred.to_csv(f"{OUT}/pred_{name}.csv", index=False)
    pd.DataFrame({"indonesian": [s for s, _ in ctrl_req], "requested": [l for _, l in ctrl_req], "output": ctrl_out,
                  "classified": [LEVELS[q] for q in p]}).to_csv(f"{OUT}/control_{name}.csv", index=False)
    return res


def run(name, extra, keep=False):
    """Resep v2 m2m_base + `extra` (pasangan ID->JV tambahan, 1x). keep=True: kembalikan model untuk generate pseudo."""
    seed_all(SEED)
    tok, model, bos = load()
    base = [(tag_src(s, l), t, l) for s, l, t in zip(tr["indonesian"], tr["level"], tr["javanese"])]
    id_pairs = [(s, t) for s, t, _ in base] + [(s, t) for s, t, l in base if l == "krama_alus"] * (OS - 1) + extra
    jv_pairs = honor_pairs()
    log(name, "pairs ID->JV", len(id_pairs), f"(pseudo {len(extra)})", "JV->JV", len(jv_pairs))

    def make_batches():
        a, b = id_pairs[:], jv_pairs[:]
        random.shuffle(a); random.shuffle(b)
        bt = [(SL, a[i:i + BS]) for i in range(0, len(a), BS)] + [(TL, b[i:i + BS]) for i in range(0, len(b), BS)]
        random.shuffle(bt); return bt

    n_batches = len(make_batches())
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * n_batches
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
    scaler = torch.amp.GradScaler("cuda")
    best, best_state, hist = -1, None, []
    for ep in range(EPOCHS):
        model.train(); tot = n = 0
        for src_lang, b in make_batches():
            tok.src_lang = src_lang
            enc = tok([s for s, _ in b], text_target=[t for _, t in b], return_tensors="pt",
                      padding=True, truncation=True, max_length=MAXLEN).to(dev)
            labels = enc.pop("labels"); labels[labels == tok.pad_token_id] = -100
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(**enc, labels=labels).logits
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100)
            opt.zero_grad(); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); tot += loss.item() * len(b); n += len(b)
        tok.src_lang = SL
        sc, c, a, alv = valid_score(translate(tok, model, bos, [tag_src(s, l) for s, l in zip(va["indonesian"], va["level"])], 1))
        hist.append({"epoch": ep + 1, "train_loss": round(tot / n, 4), "valid_chrF++": c, "valid_ctrl": a, "valid_ctrl_lv": alv, "score": round(sc, 2)})
        log(name, hist[-1])
        if sc > best:
            best = sc; best_state = {k: x.detach().to("cpu", copy=True) for k, x in model.state_dict().items()}
    model.load_state_dict(best_state); tok.src_lang = SL
    del best_state, opt, scaler, sched; gc.collect(); torch.cuda.empty_cache()

    # ---- decoding: beam4 vs rerank(lam), dipilih di valid; prediksi valid disimpan
    va_src = [tag_src(s, l) for s, l in zip(va["indonesian"], va["level"])]
    va_h = {"beam4": translate(tok, model, bos, va_src, 4)}
    va_nb = nbest(tok, model, bos, va_src)
    for lam in LAMS:
        va_h[f"rerank{lam}"] = rerank(va_nb, va["level"].tolist(), lam)
    dec = {k: valid_score(h) for k, h in va_h.items()}
    for k, v in dec.items():
        log(name, "valid decode", k, "score", round(v[0], 2), "chrF++", v[1], "ctrl", v[2], v[3])
    best_dec = max(dec, key=lambda k: dec[k][0])
    for k in ("beam4", best_dec):
        d = va.copy(); d["hyp"] = va_h[k]; d.to_csv(f"{OUT}/valid_{name}_{k}.csv", index=False)

    te_src = [tag_src(s, l) for s, l in zip(te["indonesian"], te["level"])]
    u = sorted(te["indonesian"].unique()); req = [(s, l) for s in u for l in LEVELS]
    c_src = [tag_src(s, l) for s, l in req]
    results = {"beam4": evaluate(f"{name}_beam4", translate(tok, model, bos, te_src, 4), req, translate(tok, model, bos, c_src, 4))}
    if best_dec != "beam4":
        lam = float(best_dec[len("rerank"):])
        hyp_r = rerank(nbest(tok, model, bos, te_src), te["level"].tolist(), lam)
        ctl_r = rerank(nbest(tok, model, bos, c_src), [l for _, l in req], lam)
        results[best_dec] = evaluate(f"{name}_{best_dec}", hyp_r, req, ctl_r)
    for k, r in results.items():
        log(name, k, "TEST", r["test"]["all"], "| KA", r["test"]["krama_alus"]["chrF++"],
            "| ctrl", r["control"]["level_accuracy"], r["control"]["per_level_accuracy"])

    res = {"variant": name, "n_pairs": [len(id_pairs), len(jv_pairs)], "n_pseudo": len(extra), "history": hist,
           "best_epoch": max(hist, key=lambda h: h["score"])["epoch"],
           "valid_decode": {k: {"score": round(v[0], 2), "chrF++": v[1], "ctrl": v[2], "ctrl_lv": v[3]} for k, v in dec.items()},
           "best_decode": best_dec, "test": results}
    json.dump(res, open(f"{OUT}/metrics_{name}.json", "w"), indent=1)
    if keep:
        return res, tok, model, bos
    if name != "teacher":                       # teacher = model demo yang sudah ada; tidak perlu disimpan lagi
        mdir = f"{OUT}/model_{name}"
        model.half().save_pretrained(mdir); tok.save_pretrained(mdir)
    del model; gc.collect(); torch.cuda.empty_cache()
    return res


In [ ]:
# ------------------------------------------------------------------ 1) teacher
summary = {"levels": LEVELS, "hparams": {"epochs": EPOCHS, "os": OS, "bs": BS, "lr": LR, "warmup": WARMUP, "seed": SEED,
           "maxlen": MAXLEN, "ctrl_weight": CTRL_W, "nbest": NBEST, "lams": LAMS,
           "pseudo": {"nbest": PS_NBEST, "lam": PS_LAM, "thr_p_krama_alus_reranker": PS_THR, "nusax_max_words": NUSAX_MAX_WORDS}},
           "judge": judge_info, "reranker": rr_info, "runs": {}}
res_t, tok, model, bos = run("teacher", [], keep=True)
summary["runs"]["teacher"] = res_t


In [ ]:
# ------------------------------------------------------------------ 2) pseudo-data Krama Alus dari teacher
held = set(va["indonesian"].str.lower()) | set(te["indonesian"].str.lower())
ka_groups = set(tr_raw.loc[tr_raw["level"] == "krama_alus", "group"])
pool = {"train": sorted(set(tr_raw.loc[~tr_raw["group"].isin(ka_groups), "indonesian"]))}
try:
    urllib.request.urlretrieve(NUSAX_URL, f"{OUT}/nusax_train.csv")
    nx = pd.read_csv(f"{OUT}/nusax_train.csv")["indonesian"].dropna().astype(str)
    os.remove(f"{OUT}/nusax_train.csv")          # jangan ikut tersimpan di output kernel
    train_ind = set(tr_raw["indonesian"].str.lower())
    nx = [s for s in nx if len(s.split()) <= NUSAX_MAX_WORDS and s.lower() not in held and s.lower() not in train_ind]
    pool["nusax"] = sorted(set(nx))
except Exception as e:
    log("NusaX gagal diunduh; varian nusax dilewati:", repr(e))
for k, v in pool.items():
    assert not any(s.lower() in held for s in v), "bocor ke valid/test"
    log("pool", k, len(v))


def make_pseudo(sents):
    """16 kandidat Krama Alus -> rerank (lam demo) -> simpan jika P_reranker(Krama Alus) >= PS_THR."""
    nb = nbest(tok, model, bos, [tag_src(s, "krama_alus") for s in sents], k=PS_NBEST, bs=4)
    flat = [t for c, _ in nb for t in c]; lp = rr_lp(flat); rows = []; p = 0
    for s, (c, sc) in zip(sents, nb):
        k = len(c); tot = sc + PS_LAM * lp[p:p + k, KA]; j = int(np.argmax(tot))
        rows.append({"indonesian": s, "javanese": c[j], "p_krama_alus": round(float(np.exp(lp[p + j, KA])), 4),
                     "logprob": round(float(sc[j]), 4)})
        p += k
    df = pd.DataFrame(rows)
    df["kept"] = (df["p_krama_alus"] >= PS_THR) & (df["javanese"].str.strip().str.len() > 0)
    return df


pseudo = {}
for k, sents in pool.items():
    df = make_pseudo(sents)
    pseudo[k] = df[df["kept"]]
    log("pseudo", k, "kandidat", len(df), "disimpan", int(df["kept"].sum()),
        "| rata-rata P(KA) semua", round(df["p_krama_alus"].mean(), 3), "| contoh", df[df["kept"]].head(3).values.tolist())
    # pseudo dari kalimat NusaX (CC BY-SA) disimpan terpisah supaya tidak ikut ke repo
    df.to_csv(f"{OUT}/pseudo_{k}.csv", index=False)
summary["pseudo"] = {k: {"pool": len(pool[k]), "kept": len(v), "mean_p_krama_alus_kept": round(float(v["p_krama_alus"].mean()), 4)}
                     for k, v in pseudo.items()}
del model; gc.collect(); torch.cuda.empty_cache()
json.dump(summary, open(f"{OUT}/summary_selftrain.json", "w"), indent=1, default=str)


In [ ]:
# ------------------------------------------------------------------ 3) student
for name, srcs in VARIANTS.items():
    if name == "teacher":
        continue
    if any(s not in pseudo for s in srcs):
        log("lewati", name, "(sumber pseudo tidak tersedia)"); continue
    extra = [(tag_src(s, "krama_alus"), t) for k in srcs for s, t in zip(pseudo[k]["indonesian"], pseudo[k]["javanese"])]
    try:
        summary["runs"][name] = run(name, extra)
    except Exception as e:
        import traceback; traceback.print_exc()
        summary["runs"][name] = {"error": repr(e)}
    json.dump(summary, open(f"{OUT}/summary_selftrain.json", "w"), indent=1, default=str)


In [ ]:
# ------------------------------------------------------------------ 4) pemenang (valid) dan ringkasan
ok = {k: v for k, v in summary["runs"].items() if "valid_decode" in v}
score = lambda v: v["valid_decode"][v["best_decode"]]["score"]
winner = max(ok, key=lambda k: score(ok[k]))
summary["winner"] = {"variant": winner, "decode": ok[winner]["best_decode"]}
if winner != "teacher":
    w = ok[winner]; bd = w["best_decode"]
    shutil.move(f"{OUT}/model_{winner}", f"{OUT}/model")
    cfg = {"levels": LEVELS, "tags": TAG, "src_lang": SL, "tgt_lang": TL, "seed": SEED, "variant": winner, "base_model": REPO,
           "decode": "beam4"}
    if bd != "beam4":
        rr_mod.save_pretrained(f"{OUT}/model/reranker"); rr_tok.save_pretrained(f"{OUT}/model/reranker")
        cfg.update({"decode": "rerank", "rerank_lambda": float(bd[len("rerank"):]), "nbest": NBEST, "reranker_dir": "reranker"})
    json.dump(cfg, open(f"{OUT}/model/demo_config.json", "w"), indent=1)
for k in ok:
    shutil.rmtree(f"{OUT}/model_{k}", ignore_errors=True)
json.dump(summary, open(f"{OUT}/summary_selftrain.json", "w"), indent=1, default=str)

rows = []
for k, v in ok.items():
    for d, r in v["test"].items():
        c = r["control"]
        rows.append({"varian": k, "decode": d, "valid": score(v) if d == v["best_decode"] else None,
                     "pseudo": v["n_pseudo"], "chrF++": r["test"]["all"]["chrF++"], "BLEU": r["test"]["all"]["BLEU"],
                     "chrF++ KA": r["test"]["krama_alus"]["chrF++"], "ketepatan": c["level_accuracy"],
                     "Krama Alus": c["per_level_accuracy"]["krama_alus"], "K=KA": c["identical_rate"]["krama=krama_alus"]})
log("PEMENANG (valid):", summary["winner"]); log("SELESAI")
pd.DataFrame(rows)
